In [1]:
2+2

4

In [2]:
pip install rank_bm25

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# Step 2: Global Setup & The Executive Dataset
import os
from langchain_core.documents import Document
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

print("="*60)
print(" INITIATING HYBRID SEARCH ENGINE (RRF) ")
print("="*60)

from dotenv import load_dotenv
import os
load_dotenv()

api_key=os.getenv("GEMINI_API_KEY")

# Initialize the Embedder for the Dense Engine
embedder = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

# 1. The Dataset (Executive Business News)
# Document 2 contains our exact needle. Document 1 is a semantic distractor.
mock_news_data = [
    "Q4 Financial Overview: Income pushbacks observed due to late client payments.",
    "URGENT: Project Alpha budget overrun flagged under Invoice #INV-2023-99X.",
    "IT Support Ticket: The email server is experiencing delays routing the budget.",
    "General HR Update: Remote work policies extended through Q4."
]

# Convert strings to LangChain Document objects
docs = [Document(page_content=text) for text in mock_news_data]

C:\Users\USER\AppData\Local\Temp\ipykernel_21032\3558129985.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


 INITIATING HYBRID SEARCH ENGINE (RRF) 


In [4]:
# Step 3: Initializing the Dual Engines
print("[System] Initializing Dense Vector Engine (FAISS)...")

# Engine A: The Semantic Profiler
# We embed the documents into the 768-dimensional hypersphere
vectorstore = FAISS.from_documents(docs, embedder)
faiss_retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

print("[System] Initializing Sparse Keyword Engine (BM25)...")

# Engine B: The Fingerprint Analyst
# We build a local statistical index based on Term Frequency
bm25_retriever = BM25Retriever.from_documents(docs)
bm25_retriever.k = 2

print("[System] Both engines online and awaiting fusion.")

[System] Initializing Dense Vector Engine (FAISS)...
[System] Initializing Sparse Keyword Engine (BM25)...
[System] Both engines online and awaiting fusion.


RECIPROCAL RANK FUSION

In [5]:
# Step 4: The Reciprocal Rank Fusion Execution
print("\n[System] Fusing Engines via Reciprocal Rank Fusion (RRF)...")

# The weights parameter allows us to bias the RRF algorithm.
# Setting it to [0.5, 0.5] weights the Dense and Sparse engines equally.#focus both equally if u want one to focus more increase the weight
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[0.5, 0.5]
)

print("[System] Hybrid Search Online. Commencing Stress Test...\n")

# --- EXECUTION STRESS TEST ---
# We are asking a query that requires BOTH a conceptual understanding of
# "budget overruns" AND the exact string match for the invoice number.

hybrid_query = "Find the exact budget overrun details for Invoice #INV-2023-99X."

print(f"User Query: '{hybrid_query}'\n")

# The single invoke() call triggers both databases and runs the math.
fused_results = ensemble_retriever.invoke(hybrid_query)

print("="*40)
print(" RRF FINAL RANKINGS ")
print("="*40)

for i, doc in enumerate(fused_results):
    print(f"\n--- Rank {i+1} ---")
    print(doc.page_content)


[System] Fusing Engines via Reciprocal Rank Fusion (RRF)...
[System] Hybrid Search Online. Commencing Stress Test...

User Query: 'Find the exact budget overrun details for Invoice #INV-2023-99X.'

 RRF FINAL RANKINGS 

--- Rank 1 ---
URGENT: Project Alpha budget overrun flagged under Invoice #INV-2023-99X.

--- Rank 2 ---
IT Support Ticket: The email server is experiencing delays routing the budget.
